In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from os.path import join as pjoin
from natsort import natsorted
import statsmodels.api as sm
from statsmodels.formula.api import ols
from numpy.random import RandomState, MT19937, SeedSequence

sys.path.append("../../")
import circletrack_behavior as ctb
import plotting_functions as pf

In [ ]:
## Settings
parent_dir = 'CircleTrack_Recall'
experiment_dir = 'Recall5'
exp_path = f'../../../{parent_dir}'
lin_path = f'../../../{parent_dir}/{experiment_dir}/output/lin_behav/'
circle_path = f'../../../{parent_dir}/{experiment_dir}/output/behav/'
fig_path = f'../../../{parent_dir}/{experiment_dir}/intermediate_figures'
maze_info = pd.read_csv(f'../../../{parent_dir}/{experiment_dir}/maze_yml/maze_info.csv')
chance_color = 'darkgrey'
avg_color = 'midnightblue'
subject_color = 'darkgrey'
two_group_colors = ['midnightblue', 'darkgrey']
three_group_colors = ['darkorchid', 'midnightblue', 'darkgrey']
four_group_colors = ['darkorchid', 'midnightblue', 'orchid', 'blue']
group_colors_dict = {'Recall A': 'midnightblue', 'B Recall A': 'darkorchid', 'Recall B': 'darkgrey'}
error_dict = {'Recall A': 'rgba(0,41,102,0.4)', 'B Recall A': 'rgba(153,50,204,0.4)',
              'Recall B': 'rgba(169,169,169,0.4)'}
three_color_plots = ['#7d7d7d', 'blue', "#28b3c9"]
group_colors_dict = {'5 day': '#7d7d7d', '10 day': 'blue', '15 day': "#28b3c9"}
symbol_list = ['circle', 'circle', 'circle']
excluded_mice = ['mcr95'] ## mcr95 on day 5 water ports weren't working, so extinguished behavior
excluded_mice_both = ['mcr09', 'mcr16', 'mcr95'] ## excluded mice from Recall1 and Recall5

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

### Circle track lick accuracy and rewards.

In [ ]:
circletrack_results = {'mouse': [], 'day': [], 'sex': [], 'group': [], 'session': [], 'lick_accuracy': [], 'rewards': []}
for mouse in os.listdir(circle_path):
    mouse_path = pjoin(circle_path, mouse)
    sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
    group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
    for idx, session in enumerate(os.listdir(mouse_path)):
        behav = pd.read_feather(pjoin(mouse_path, f'{session}'))
        behav = behav[~behav['probe']]
        reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]
        pc_thresh5 = ctb.lick_accuracy(behav, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
        circletrack_results['mouse'].append(mouse)
        circletrack_results['day'].append(idx+1)
        circletrack_results['sex'].append(sex)
        circletrack_results['group'].append(group)
        circletrack_results['session'].append(np.unique(behav['session'])[0])
        circletrack_results['lick_accuracy'].append(pc_thresh5)
        circletrack_results['rewards'].append(np.sum(behav['water']))
ct_df = pd.DataFrame(circletrack_results)

In [ ]:
## Plot 5th lick accuracy across days
fig = pf.plot_behavior_across_days(ct_df, x_var='day', y_var='lick_accuracy', groupby_var=['day'], plot_transitions=[5.5], transition_color=['darkgrey'],
                                   marker_color=avg_color, avg_color=avg_color, expert_line=False, chance=True, symbols=['circle'],
                                   x_title='Day', y_title='Lick Accuracy (%)', titles=['Circle Track'], height=500, width=500)
fig.update_yaxes(range=[0, 100])
fig.show()

In [ ]:
## Plot rewards across days
fig = pf.plot_behavior_across_days(ct_df, x_var='day', y_var='rewards', groupby_var=['day'], plot_transitions=[5.5], transition_color=['darkgrey'],
                                   marker_color=avg_color, avg_color=avg_color, expert_line=False, chance=False, symbols=['circle'],
                                   x_title='Day', y_title='Rewards', titles=['Circle Track'], height=500, width=500)
fig.show()

In [ ]:
## Separate by the three groups for first five days
subset_df = ct_df.copy()
subset_df = subset_df.replace({'five_day': '5 day', 'ten_day': '10 day', 'fifteen_day': '15 day'})
avg = subset_df.groupby(['group', 'day'], as_index=False).agg({'lick_accuracy': ['mean', 'sem']})
fig = pf.custom_graph_template(x_title='Day', y_title='Lick Accuracy (%)', titles=['Training'])
for group in ['5 day', '10 day', '15 day']:
    gdata = avg[(avg['day'] < 6) & (avg['group'] == group)]
    fig.add_trace(go.Scatter(x=gdata['day'], y=gdata['lick_accuracy']['mean'], mode='lines+markers', line_color=group_colors_dict[group],
                             name=group, error_y=dict(type='data', array=gdata['lick_accuracy']['sem'], thickness=2),
                             marker_size=6, marker=dict(line=dict(width=1.5))))
fig.update_yaxes(range=[0, 100])
fig.add_hline(y=25, line_width=3, opacity=1, line_dash='dash', line_color='darkgrey')
fig.show()
fig.write_image(pjoin(fig_path, 'three_group_lick_accuracyuracy.png'), width=500, height=500)

In [ ]:
## Separate by the three groups for all six days
subset_df = ct_df.copy()
subset_df = subset_df.replace({'five_day': '5 day', 'ten_day': '10 day', 'fifteen_day': '15 day'})
avg = subset_df.groupby(['group', 'day'], as_index=False).agg({'lick_accuracy': ['mean', 'sem']})
fig = pf.custom_graph_template(x_title='Day', y_title='Lick Accuracy (%)', titles=['Training'])
for group in ['5 day', '10 day', '15 day']:
    gdata = avg[avg['group'] == group]
    fig.add_trace(go.Scatter(x=gdata['day'], y=gdata['lick_accuracy']['mean'], mode='lines+markers', line_color=group_colors_dict[group],
                             name=group, error_y=dict(type='data', array=gdata['lick_accuracy']['sem'], thickness=2),
                             marker_size=6, marker=dict(line=dict(width=1.5))))
fig.update_yaxes(range=[0, 100])
fig.add_hline(y=25, line_width=3, opacity=1, line_dash='dash', line_color='darkgrey')
fig.add_vline(x=5.5, line_width=3, opacity=1, line_dash='dash', line_color='darkgrey')
fig.show()

### Look at probe accuracy.

In [ ]:
## Excluding mice
lick_dict_probe = {'mouse': [], 'group': [], 'sex': [], 'session': [], 'day': [], 'num_licks': [], 'probe_acc': [], 'session_acc': []}
for mouse in os.listdir(circle_path):
    mpath = pjoin(circle_path, mouse)
    sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
    group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
    if mouse in excluded_mice:
        pass
    else:
        for idx, session in enumerate(natsorted(os.listdir(mpath))):
            behav = pd.read_feather(pjoin(mpath, session))
            if any(behav['probe']):
                behav_probe = behav[behav['probe']]
                behav_no_probe = behav[~behav['probe']]
                reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]
                percent_correct = ctb.lick_accuracy(behav_probe, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
                session_pc = ctb.lick_accuracy(behav_no_probe, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
                lick_dict_probe['mouse'].append(mouse)
                lick_dict_probe['group'].append(group)
                lick_dict_probe['sex'].append(sex)
                lick_dict_probe['day'].append(idx+1)
                lick_dict_probe['session'].append(np.unique(behav['session'])[0])
                lick_dict_probe['num_licks'].append(len(behav_probe[behav_probe['lick_port'] != -1]))
                lick_dict_probe['probe_acc'].append(percent_correct)
                lick_dict_probe['session_acc'].append(session_pc)
            else:
                pass
probe_df = pd.DataFrame(lick_dict_probe)
probe_df = probe_df.replace({'five_day': '5 day', 'ten_day': '10 day', 'fifteen_day': '15 day'})
avg_group_probe = probe_df.groupby(['group', 'day'], as_index=False).agg({'probe_acc': ['mean', 'sem']})

In [ ]:
## Plot probe performance for the three created groups
fig = pf.custom_graph_template(x_title='Day', y_title='', titles=[''], 
                               shared_x=True, shared_y=True, height=500, width=600, font_size=26)
for group in ['5 day', '10 day', '15 day']:
    gdata = avg_group_probe[avg_group_probe['group'] == group]
    fig.add_trace(go.Scatter(x=gdata['day'], y=gdata['probe_acc']['mean'], mode='markers',
                             error_y=dict(type='data', array=gdata['probe_acc']['sem'], thickness=2), 
                             line_color=group_colors_dict[group], name=group, showlegend=True, legendgroup=group))

fig.add_hline(y=25, line_width=3, line_dash='dash', line_color='darkgrey', opacity=1)
fig.add_vline(x=5.5, line_width=3, line_dash='dash', line_color='darkgrey', opacity=1)
fig.update_yaxes(title='Lick Accuracy (%)', col=1, range=[0, 100])
fig.update_xaxes(
    ticktext=['1', '5', 'Recall'],
    tickvals=[1, 5, 6],
)
fig.show()
fig.write_image(pjoin(fig_path, 'probe_accuracy_recall.png'), width=600, height=500)

### Combine across Recall1 and Recall5.

In [ ]:
circletrack_results = {'mouse': [], 'day': [], 'group': [], 'session': [], 'lick_accuracy': [], 'rewards': []}
for exp in ['Recall1', 'Recall5']:
    dpath = pjoin(exp_path, f'{exp}/output/behav')
    maze_info = pd.read_csv(f'../../../{parent_dir}/{exp}/maze_yml/maze_info.csv')
    for mouse in natsorted(os.listdir(dpath)):
        if mouse in excluded_mice_both:
            pass
        else:
            mpath = pjoin(dpath, mouse)
            group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
            for idx, session in enumerate(natsorted(os.listdir(mpath))):
                behav = pd.read_feather(pjoin(mpath, f'{session}'))
                behav = behav[~behav['probe']]
                reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]
                pc_thresh5 = ctb.lick_accuracy(behav, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
                circletrack_results['mouse'].append(mouse)
                circletrack_results['day'].append(idx+1)
                circletrack_results['group'].append(group)
                circletrack_results['session'].append(np.unique(behav['session'])[0])
                circletrack_results['lick_accuracy'].append(pc_thresh5)
                circletrack_results['rewards'].append(np.sum(behav['water']))
ct_df = pd.DataFrame(circletrack_results)

In [ ]:
ct_df.to_feather('../../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data/recall_ct_df.feat')

In [ ]:
## Separate by the three groups for first five days for lick accuracy
subset_df = ct_df.copy()
subset_df = subset_df.replace({'five_day': '5 day', 'ten_day': '10 day', 'fifteen_day': '15 day'})
avg = subset_df.groupby(['group', 'day'], as_index=False).agg({'lick_accuracy': ['mean', 'sem']})
fig = pf.custom_graph_template(x_title='Day', y_title='Lick Accuracy (%)', titles=[''])

for mouse in subset_df['mouse'].unique():
    mdata = subset_df[(subset_df['mouse'] == mouse) & (subset_df['day'] < 6)]
    fig.add_trace(go.Scattergl(x=mdata['day'], y=mdata['lick_accuracy'], mode='lines', line_color=group_colors_dict[mdata['group'].unique()[0]],
                               showlegend=False, name=mouse, line_width=1, opacity=0.4))
for group in ['5 day', '10 day', '15 day']:
    gdata = avg[(avg['day'] < 6) & (avg['group'] == group)]
    fig.add_trace(go.Scattergl(x=gdata['day'], y=gdata['lick_accuracy']['mean'], mode='lines+markers', line_color=group_colors_dict[group],
                             name=group, error_y=dict(type='data', array=gdata['lick_accuracy']['sem'], thickness=3.5),
                             marker_size=11, marker=dict(line=dict(width=2.5))))
fig.update_yaxes(range=[-2, 102])
fig.add_hline(y=25, line_width=3, opacity=1, line_dash='dash', line_color='darkgrey')
fig.show()
fig.write_image(pjoin(fig_path, 'lick_accuracy_both_experiments.svg'), width=500, height=500)

In [ ]:
## Separate by the three groups for first five days for rewards
subset_df = ct_df.copy()
subset_df = subset_df.replace({'five_day': '5 day', 'ten_day': '10 day', 'fifteen_day': '15 day'})
avg = subset_df.groupby(['group', 'day'], as_index=False).agg({'rewards': ['mean', 'sem']})
fig = pf.custom_graph_template(x_title='Day', y_title='Rewards', titles=[''])

for mouse in subset_df['mouse'].unique():
    mdata = subset_df[(subset_df['mouse'] == mouse) & (subset_df['day'] < 6)]
    fig.add_trace(go.Scattergl(x=mdata['day'], y=mdata['rewards'], mode='lines', line_color=group_colors_dict[mdata['group'].unique()[0]],
                               showlegend=False, name=mouse, line_width=1, opacity=0.4))
for group in ['5 day', '10 day', '15 day']:
    gdata = avg[(avg['day'] < 6) & (avg['group'] == group)]
    fig.add_trace(go.Scattergl(x=gdata['day'], y=gdata['rewards']['mean'], mode='lines+markers', line_color=group_colors_dict[group],
                             name=group, error_y=dict(type='data', array=gdata['rewards']['sem'], thickness=3.5),
                             marker_size=11, marker=dict(line=dict(width=2.5))))
# fig.update_yaxes(range=[0, 130])
fig.show()
fig.write_image(pjoin(fig_path, 'rewards_both_experiments.svg'), width=500, height=500)

In [ ]:
lick_dict_probe = {'mouse': [], 'group': [], 'session': [], 'day': [], 'num_licks': [], 'probe_acc': [], 'session_acc': []}
for exp in ['Recall1', 'Recall5']:
    dpath = pjoin(exp_path, f'{exp}/output/behav')
    maze_info = pd.read_csv(f'../../../{parent_dir}/{exp}/maze_yml/maze_info.csv')
    for mouse in natsorted(os.listdir(dpath)):
        if mouse in excluded_mice_both:
            pass
        else:
            mpath = pjoin(dpath, mouse)
            group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
            for idx, session in enumerate(natsorted(os.listdir(mpath))):
                behav = pd.read_feather(pjoin(mpath, session))
                if any(behav['probe']):
                    behav_probe = behav[behav['probe']]
                    behav_no_probe = behav[~behav['probe']]
                    reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]
                    percent_correct = ctb.lick_accuracy(behav_probe, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
                    session_pc = ctb.lick_accuracy(behav_no_probe, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
                    lick_dict_probe['mouse'].append(mouse)
                    lick_dict_probe['group'].append(group)
                    lick_dict_probe['day'].append(idx+1)
                    lick_dict_probe['session'].append(np.unique(behav['session'])[0])
                    lick_dict_probe['num_licks'].append(len(behav_probe[behav_probe['lick_port'] != -1]))
                    lick_dict_probe['probe_acc'].append(percent_correct)
                    lick_dict_probe['session_acc'].append(session_pc)
probe_df = pd.DataFrame(lick_dict_probe)
probe_df = probe_df.replace({'five_day': '5 day', 'ten_day': '10 day', 'fifteen_day': '15 day'})
avg_group_probe = probe_df.groupby(['group', 'day'], as_index=False).agg({'probe_acc': ['mean', 'sem']})

In [ ]:
probe_df.to_feather('../../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data/recall_probe_df.feat')

In [ ]:
## Plot probe performance for the three created groups
fig = pf.custom_graph_template(x_title='Day', y_title='', titles=[''], 
                               shared_x=True, shared_y=True, height=500, width=600)

rs = RandomState(MT19937(SeedSequence(10)))
shifts = rs.uniform(-0.25, 0.25, (probe_df['mouse'].unique().shape[0], 3))

for idx, mouse in enumerate(probe_df['mouse'].unique()):
    mdata = probe_df[probe_df['mouse'] == mouse]
    if mdata.shape[0] == 1:
        sh = shifts[idx][0]
    elif mdata.shape[0] == 2:
        sh = shifts[idx][:2]
    else:
        sh = shifts[idx]
    fig.add_trace(go.Scattergl(x=mdata['day'] + sh, y=mdata['probe_acc'], mode='markers', marker_size=9,
                               marker=dict(line=dict(width=2.5, color=group_colors_dict[mdata['group'].unique()[0]])), name=mouse, showlegend=False,
                               marker_color='rgba(0,0,0,0)'))

for group in ['5 day', '10 day', '15 day']:
    gdata = avg_group_probe[avg_group_probe['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['day'], y=gdata['probe_acc']['mean'], mode='markers', marker_size=11,
                             marker=dict(line=dict(width=2.5, color='black')),
                             error_y=dict(type='data', array=gdata['probe_acc']['sem'], thickness=3.5), 
                             line_color=group_colors_dict[group], name=group, showlegend=True, legendgroup=group))

fig.add_hline(y=25, line_width=3, line_dash='dash', line_color='darkgrey', opacity=1)
fig.add_vline(x=5.5, line_width=3, line_dash='dash', line_color='darkgrey', opacity=1)
fig.update_yaxes(title='Lick Accuracy (%)', col=1, range=[-2, 102])
fig.update_xaxes(
    tickvals=[1, 5, 6],
    ticktext=['1', '5', 'Recall'],
)
fig.show()
fig.write_image(pjoin(fig_path, f'combined_probe_accuracy_recall_3groups_wlegend.svg'), width=600, height=500)